In [28]:
import pandas as pd
from dataclasses import dataclass, asdict
from ml_enhance import QuantumFPFileLoader, QuantumFPDatasetBuilder, ConformerAggregator

## GFN2-xTB comparison to QM9 values

QM9 is a dataset consisting of about 130k molecules, for which QM properties are calculated on the DFT level of theory. Exact specifications of the level of theory are the following: **B3LYP/6-31G(2df,p)** (hybrid-GGA)

GFN2-xTB is a semi-empirical method that uses a parameterized density functional tight-binding method on the LDA level of theory using a minimal STO-nG basis set. This method also adds a correction to the electron density $\rho$ by accounting for up to the third order variation in the electron density $\delta \rho$.


Some of the molecular properties that are both present in the QM9 dataset and are generated by QuantumFP

|                          | QM9 unit |         QFP unit         |
|:------------------------:|:--------:|:------------------------:|
|   Dipole moment (norm)   |   Debye  |        $e$ $a_0$         |
| Isotropic polarizability |  $a_0^3$ | $e^2$ $a_0^2$ / hartree  |
|       HOMO-LUMO gap      |    eV    |          hartree         |
|     Zero point energy    |    eV    |          kcal/mol        |
|    Atomization energy    |    eV    |          hartree         |

In [95]:
@dataclass
class ConversionUnits:
    """Class containing conversion units from QM9 units to QFP units."""
    mu: float = 0.3934303 # Debye to e bohr
    alpha: float = 1.0
    gap: float = 0.0367493 # eV to hartree
    zpve: float = 23.0605 # eV to kcal/mol
    U0_atom: float = 0.0367493 # eV to hartree

conversion = asdict(ConversionUnits())

In [96]:
loader = QuantumFPFileLoader("../data/QM9")
files = loader.list_output_files()

In [97]:
wanted_properties = ["original_smiles", "energy", "molecular_dipole_norm", "molecular_polarizability_mean", "homo_lumo_gap", "zero_point_energy", "atomization_energy"]

In [144]:
pd.read_csv("../data/processed_dataset.csv")["molecular_polarizability_mean"]

0       -99.744189
1       -95.248037
2       -98.287723
3       -90.301117
4      -230.053983
           ...    
8832   -175.035008
8833   -210.040379
8834   -244.342833
8835   -112.376440
8836   -128.870277
Name: molecular_polarizability_mean, Length: 8837, dtype: float64

In [168]:
l = []
for file in files:
    for df in loader.stream_conformer_dataframe(file):
        tdf = df.loc[df["energy"].argmin(), wanted_properties]
        l.append(tdf)

qfp_data = pd.DataFrame(l).reset_index(drop=True)
qfp_data.head()

,original_smiles,energy,molecular_dipole_norm,molecular_polarizability_mean,homo_lumo_gap,zero_point_energy,atomization_energy
0,[O:1]([C:2]([C@@:3]1([O:4][H:13])[C:5]([H:14])...,-30.618349,1.149334,-46.740433,0.306714,125.673792,4.171872
1,[O:1]([C:2]([c:3]1[n:4][c:5]([N:6]([H:13])[H:1...,-28.409087,0.364225,-47.956684,0.150365,83.239035,3.861726
2,[C:1]1([H:9])([H:10])[O:2][c:3]2[c:4]([o:5][n:...,-24.509994,2.134389,-38.715796,0.122238,52.483196,2.991961
3,[C:1]1([H:9])=[C:2]2[C:3]([H:10])([H:11])[C:4]...,-25.027677,1.430345,-42.955065,0.165460,80.671584,3.559774
4,[N:1]([C:2](=[O:3])[C@:4]1([H:12])[C:5]([H:13]...,-28.459354,1.453168,-45.951560,0.178691,90.397975,3.986876


In [183]:
qfp_data = qfp_data.rename(columns={"original_smiles": "ref_smiles"})
qfp_data["molecular_polarizability_mean"] = abs(qfp_data["molecular_polarizability_mean"])
qfp_data.head()

,ref_smiles,energy,molecular_dipole_norm,molecular_polarizability_mean,homo_lumo_gap,zero_point_energy,atomization_energy
0,[O:1]([C:2]([C@@:3]1([O:4][H:13])[C:5]([H:14])...,-30.618349,1.149334,46.740433,0.306714,125.673792,4.171872
1,[O:1]([C:2]([c:3]1[n:4][c:5]([N:6]([H:13])[H:1...,-28.409087,0.364225,47.956684,0.150365,83.239035,3.861726
2,[C:1]1([H:9])([H:10])[O:2][c:3]2[c:4]([o:5][n:...,-24.509994,2.134389,38.715796,0.122238,52.483196,2.991961
3,[C:1]1([H:9])=[C:2]2[C:3]([H:10])([H:11])[C:4]...,-25.027677,1.430345,42.955065,0.165460,80.671584,3.559774
4,[N:1]([C:2](=[O:3])[C@:4]1([H:12])[C:5]([H:13]...,-28.459354,1.453168,45.951560,0.178691,90.397975,3.986876


For each molecule, we have loaded collected the minimal energy conformer and will compare its properties to the QM9. **This comparison does however come with an inconsistency in the fact that the molecular properties are calculated on different molecular geometries**.

Geometry optimization in QM9 consists of an initial relaxation using PM7 semi-empirical method, followed by a B3LYP relaxation. (see QM9 paper)

Geometry optimization in QuantumFP consists of an initial relaxation at the GFN-FF level of theory, followed by a GFN2-xTB minimization.

Despite this discrepancy, this comparison still provides a crude, general indication of how accurate QuantumFP features are compared to DFT level of theory.

In [170]:
name_map = {
    "U0_atom": "atomization_energy",
    "mu": "molecular_dipole_norm",
    "alpha": "molecular_polarizability_mean",
    "gap": "homo_lumo_gap",
    "zpve": "zero_point_energy"
}

In [186]:
qm9_data = pd.read_csv("qm9_values_of_mols.csv").rename(columns={"cleaned_smiles": "ref_smiles"})
qm9_data["U0_atom"] = abs(qm9_data["U0_atom"])
qm9_data.head()

,Unnamed: 0,smiles,mu,alpha,gap,zpve,U0_atom,ref_smiles
0,101593,[H]OC([H])([H])[C@@]1(O[H])C([H])([H])[N@@H+](...,2.1006,79.139999,7.902186,5.227470,86.053978,[O:1]([C:2]([C@@:3]1([O:4][H:13])[C:5]([H:14])...
1,125072,[H]OC([H])([H])C1=NC(N([H])[H])=NN1C([H])([H])[H],1.5036,72.180000,5.842285,3.747607,69.721405,[O:1]([C:2]([c:3]1[n:4][c:5]([N:6]([H:13])[H:1...
2,34541,[H]N1C([H])([H])C([H])([H])[C@]2([H])O[C@]([H]...,5.2682,71.320000,6.783799,3.774137,73.071831,[N:1]1([H:10])[C:2]([H:11])([H:12])[C:3]([H:13...
3,27170,[H]C(=O)C1=C(N([H])[H])OC(N([H])[H])=N1,6.1144,70.970001,4.772877,2.761928,62.040253,[C:1](=[O:2])([c:3]1[c:4]([N:5]([H:11])[H:12])...
4,103692,[H]N(C([H])([H])[H])[C@]1(C#N)C([H])([H])OC([H...,3.4547,75.580002,7.681774,4.365822,78.730164,[N:1]([C:2]([H:11])([H:12])[H:13])([C@:3]1([C:...


In [187]:
for col in qm9_data.columns: 
    if col in conversion.keys():
        qm9_data[f"{name_map[col]}_conv"] = qm9_data[col] * conversion[col]
    else:
        print("not converted: ", col)

not converted:  Unnamed: 0
not converted:  smiles
not converted:  ref_smiles


In [188]:
conv_qm9_data = qm9_data.filter(regex="ref|conv")

In [189]:
merged = pd.merge(qfp_data, conv_qm9_data, on="ref_smiles")
merged = merged.reindex(sorted(merged.columns), axis=1)

In [190]:
merged.head()

,atomization_energy,atomization_energy_conv,energy,homo_lumo_gap,homo_lumo_gap_conv,molecular_dipole_norm,molecular_dipole_norm_conv,molecular_polarizability_mean,molecular_polarizability_mean_conv,ref_smiles,zero_point_energy,zero_point_energy_conv
0,4.171872,3.162423,-30.618349,0.306714,0.2904,1.149334,0.826440,46.740433,79.139999,[O:1]([C:2]([C@@:3]1([O:4][H:13])[C:5]([H:14])...,125.673792,120.548081
1,3.861726,2.562213,-28.409087,0.150365,0.2147,0.364225,0.591562,47.956684,72.180000,[O:1]([C:2]([c:3]1[n:4][c:5]([N:6]([H:13])[H:1...,83.239035,86.421680
2,2.991961,1.927832,-24.509994,0.122238,0.1841,2.134389,1.930130,38.715796,56.410000,[C:1]1([H:9])([H:10])[O:2][c:3]2[c:4]([o:5][n:...,52.483196,53.341960
3,3.559774,2.527295,-25.027677,0.165460,0.2510,1.430345,1.183242,42.955065,66.440002,[C:1]1([H:9])=[C:2]2[C:3]([H:10])([H:11])[C:4]...,80.671584,83.326184
4,3.986876,2.795556,-28.459354,0.178691,0.2624,1.453168,1.279868,45.951560,71.959999,[N:1]([C:2](=[O:3])[C@:4]1([H:12])[C:5]([H:13]...,90.397975,92.229899


In [203]:
import numpy as np
suffix = "_conv"

nmae_results = {}

for col in merged.columns:
    if col.endswith(suffix):
        base = col.replace(suffix, "")
        if base in merged.columns:
            y_true = merged[col]
            y_pred = merged[base]

            mae = (y_true - y_pred).abs().mean()
            std = np.std(y_true)

            nmae = mae / std
            nmae_results[base] = nmae

nmae_series = pd.Series(nmae_results).sort_values()
print(nmae_series)

zero_point_energy                0.208616
molecular_dipole_norm            0.949429
homo_lumo_gap                    1.763504
atomization_energy               3.517352
molecular_polarizability_mean    4.110409
dtype: float64


|nMAE | Meaning |
|-----|--------------- |
|~0.1 | excellent |
|~0.2–0.5 | good/moderate |
|~1.0 | error comparable to dataset variability |
|>1.0 | poor / uninformative model |